# Donut Fine-Tuning for SurveyPlan AI
This notebook is specifically configured to fine-tune the `naver-clova-ix/donut-base` model on survey plan crops.

**Recent Updates:**
- Unified `az_dist` support
- Added tokens for `<curve_data>`, `<street>`, and `<title_data>`
- Automatic re-fetching of latest repository code

In [ ]:
# Update your repository code if running in Colab
!git pull origin main

## 1. Environment & Setup

In [ ]:
!pip install -q transformers datasets pytorch-lightning sentencepiece torchvision accelerate editdistance

from google.colab import drive
drive.mount('/content/drive')

## Phase I: Data Engineering Logic

In [ ]:
import os
import json
import re
from pathlib import Path

# PATH UPDATED TO RUNS/DONUT_TUNING
DATA_DIR = Path('/content/drive/MyDrive/SurveyPlan AI/runs/donut_tuning')
os.makedirs(DATA_DIR, exist_ok=True)

# Normalization Utility
def normalize_survey_text(text: str) -> str:
    """
    Cleans survey-specific characters.
    Example: 45° 30' 15" -> 45- 30. 15.
    """
    text = text.replace('°', '-')
    text = text.replace("'", '.')
    text = text.replace('"', '.')
    return text

print("Path and Normalization logic initialized.")

In [ ]:
from datasets import load_dataset

# Attempt to load from the specified DATA_DIR
# metadata.jsonl AND image files must be in this folder
try:
    dataset = load_dataset("imagefolder", data_dir=str(DATA_DIR), split="train")
    dataset = dataset.train_test_split(test_size=0.1)
    train_dataset = dataset['train']
    val_dataset = dataset['test']
    print(f"SUCCESS: Loaded {len(train_dataset)} train and {len(val_dataset)} validation samples.")
except Exception as e:
    print(f"ERROR LOADING DATASET: {e}")

## Phase II: Training Pipeline

In [ ]:
from transformers import DonutProcessor, VisionEncoderDecoderModel, VisionEncoderDecoderConfig
import torch

model_id = "naver-clova-ix/donut-base"

# 1. Processor Setup with custom crop size for high fidelity text
processor = DonutProcessor.from_pretrained(model_id)
processor.image_processor.size = {"height": 1280, "width": 960}

# 2. Add custom alphabet / schema tokens (UPDATED)
task_start_tokens = ["<s_lot_geometry>", "<s_parcel_info>", "<s_tabular_data>", "<s_general>"]
schema_tokens = [
    "<az>", "<dist>", "<curve_data>", "<street>", "<title_data>", 
    "<lot_id>", "<adj_id>", "<area_val>",
    "<row>", "<pt_id>", "<north>", "<east>", "<radius>", "<arc>", "<delta>", "<chord>",
    "<plan_title>", "<text>"
]

new_tokens = task_start_tokens + schema_tokens
for token in schema_tokens:
    new_tokens.append(token.replace("<", "</"))

processor.tokenizer.add_tokens(new_tokens)

# 3. Model Customization
model = VisionEncoderDecoderModel.from_pretrained(model_id)
model.decoder.resize_token_embeddings(len(processor.tokenizer))

model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(["<s_general>"])[0] 
model.config.eos_token_id = processor.tokenizer.eos_token_id

print(f"Vocabulary resized and tokens added. Tokenizer size: {len(processor.tokenizer)}")

In [ ]:
from torch.utils.data import Dataset, DataLoader

def json2token(obj, sort_json_key: bool = True):
    """
    Converts a JSON object (dict) into a token sequence for training.
    """
    if isinstance(obj, dict):
        if len(obj) == 1 and "text_sequence" in obj:
            return obj["text_sequence"]
        output = ""
        keys = sorted(obj.keys()) if sort_json_key else obj.keys()
        for k in keys:
            output += f"<{k}>"
            output += json2token(obj[k], sort_json_key)
            output += f"</{k}>"
        return output
    elif isinstance(obj, list):
        return "".join([json2token(item, sort_json_key) for item in obj])
    else:
        return str(obj)

class DonutDataset(Dataset):
    def __init__(self, dataset, processor, max_length=512, split="train", task_prompt="<s_general>"):
        super().__init__()
        self.dataset = dataset
        self.processor = processor
        self.max_length = max_length
        self.split = split
        self.task_prompt = task_prompt

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image'].convert('RGB')
        
        gt = item['ground_truth']
        if isinstance(gt, str): gt = json.loads(gt)
        gt_parse = gt.get('gt_parse', gt)
        
        # Automatically select prompt based on root key
        prompt = self.task_prompt
        if isinstance(gt_parse, dict) and len(gt_parse) == 1:
            root_key = list(gt_parse.keys())[0]
            if f"<s_{root_key}>" in task_start_tokens:
                prompt = f"<s_{root_key}>"
                
        # FIX: json2token (converts dict -> string)
        sequence = prompt + json2token(gt_parse) + self.processor.tokenizer.eos_token
        
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze()
        labels = self.processor.tokenizer(
            sequence, add_special_tokens=False, 
            max_length=self.max_length, padding="max_length", 
            truncation=True, return_tensors="pt"
        )["input_ids"].squeeze()
        
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

train_ds = DonutDataset(train_dataset, processor)
val_ds = DonutDataset(val_dataset, processor, split="val")
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)

In [ ]:
import pytorch_lightning as pl
import editdistance

class DonutModelPLModule(pl.LightningModule):
    def __init__(self, model, processor, learning_rate=3e-5):
        super().__init__()
        self.model = model
        self.processor = processor
        self.learning_rate = learning_rate

    def training_step(self, batch, batch_idx):
        outputs = self.model(pixel_values=batch["pixel_values"], labels=batch["labels"])
        loss = outputs.loss
        self.log("train_loss", loss)
        return loss

    def validation_step(self, batch, batch_idx):
        pixel_values = batch["pixel_values"]
        labels = batch["labels"].clone()
        labels[labels == -100] = self.processor.tokenizer.pad_token_id
        
        prompt_ids = labels[:, :1] 
        outputs = self.model.generate(
            pixel_values, decoder_input_ids=prompt_ids,
            max_length=self.model.decoder.config.max_position_embeddings,
            pad_token_id=self.processor.tokenizer.pad_token_id,
            eos_token_id=self.processor.tokenizer.eos_token_id,
            use_cache=True, return_dict_in_generate=True,
        )
        
        preds = self.processor.tokenizer.batch_decode(outputs.sequences, skip_special_tokens=True)
        gts = self.processor.tokenizer.batch_decode(labels, skip_special_tokens=True)
        
        for pred, gt in zip(preds, gts):
            dist = editdistance.eval(pred.strip(), gt.strip())
            self.log("val_edit_distance", float(dist))
        return 0

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.learning_rate)

pl_module = DonutModelPLModule(model, processor)
trainer = pl.Trainer(max_epochs=100, accelerator="gpu", precision="16-mixed")

print("Training STARTING...")
trainer.fit(pl_module, train_loader, val_loader)

In [ ]:
EXPORT_DIR = '/content/drive/MyDrive/SurveyPlan AI/Models/Donut_Finetuned'
os.makedirs(EXPORT_DIR, exist_ok=True)

print(f"Saving to {EXPORT_DIR}...")
model.save_pretrained(EXPORT_DIR)
processor.save_pretrained(EXPORT_DIR)
print("Success!")